# E-Commerce Logistics & Delivery Performance Analysis

**Author:** Marziyeh Eslamparasti — Business Analyst, Hamburg  
**Dataset:** [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Tools:** Python · Pandas · DuckDB (SQL) · Scikit-learn · Matplotlib

I used 96,470 delivered orders from 2016–2018 to examine delivery reliability, customer reviews, category and regional differences, and late-delivery risk.

## Questions I looked at

1. How reliable was delivery performance?
2. Which categories and seller states were associated with weaker results?
3. How did lateness relate to review scores?
4. Can information available at order time help screen for late-delivery risk?
5. How did order volume, revenue, and service quality change over time?

> The results are descriptive. They show patterns in this historical dataset, not current market performance or proof of causation.

## 1. Setup and data validation

Download the seven Olist CSV files listed in [`data/README.md`](../data/README.md) and place them in `data/raw/`. The setup cell works whether Jupyter starts from the repository root or from the `notebooks` folder.

In [ ]:
from pathlib import Path
import sys
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)


def find_project_root(start: Path) -> Path:
    # Locate the repository root from the current working directory.
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import best_f1_threshold, calculate_delivery_delay, summarise_performance

FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "categories": "product_category_name_translation.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
}

missing_files = [name for name in FILES.values() if not (DATA_DIR / name).exists()]
if missing_files:
    missing = "\n- ".join(missing_files)
    raise FileNotFoundError(
        f"Missing source files in {DATA_DIR}:\n- {missing}\n"
        "See data/README.md for download instructions."
    )

print(f"Project root: {PROJECT_ROOT}")
print("All required source files are available.")

In [ ]:
raw_orders = pd.read_csv(
    DATA_DIR / FILES["orders"],
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
raw_items = pd.read_csv(DATA_DIR / FILES["items"])
raw_reviews = pd.read_csv(DATA_DIR / FILES["reviews"])
raw_products = pd.read_csv(DATA_DIR / FILES["products"])
category_names = pd.read_csv(DATA_DIR / FILES["categories"])
raw_sellers = pd.read_csv(DATA_DIR / FILES["sellers"])
raw_payments = pd.read_csv(DATA_DIR / FILES["payments"])

print(f"Orders loaded:      {len(raw_orders):,}")
print(f"Order items loaded: {len(raw_items):,}")
print(f"Reviews loaded:     {len(raw_reviews):,}")

## 2. Prepare the order-level dataset

Only delivered orders with actual and estimated delivery dates are used for delivery-performance metrics.

`delivery_delay_days = actual delivery date − estimated delivery date`

- Negative: delivered early
- Zero: delivered on the promised date
- Positive: delivered late

For an order with multiple items or sellers, I use the first product and seller as descriptive attributes while keeping the order totals. This is a simplification; a production model should represent multi-seller orders explicitly.

In [ ]:
delivered_orders = raw_orders.loc[raw_orders["order_status"].eq("delivered")].copy()
delivered_orders["delivery_delay_days"] = calculate_delivery_delay(
    delivered_orders,
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
)
delivered_orders["on_time"] = delivered_orders["delivery_delay_days"].le(0).astype(int)
delivered_orders["late"] = 1 - delivered_orders["on_time"]
delivered_orders["order_month"] = (
    delivered_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
delivered_orders["order_quarter"] = (
    delivered_orders["order_purchase_timestamp"].dt.to_period("Q").astype(str)
)
delivered_orders["day_of_week"] = delivered_orders["order_purchase_timestamp"].dt.dayofweek
delivered_orders["month_num"] = delivered_orders["order_purchase_timestamp"].dt.month
delivered_orders["promised_lead_days"] = (
    delivered_orders["order_estimated_delivery_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86_400

items_per_order = (
    raw_items.groupby("order_id")
    .agg(
        item_price_total=("price", "sum"),
        freight_value_total=("freight_value", "sum"),
        item_count=("order_item_id", "max"),
        product_id=("product_id", "first"),
        seller_id=("seller_id", "first"),
    )
    .reset_index()
)
review_per_order = raw_reviews.groupby("order_id")["review_score"].first().reset_index()
products_with_category = raw_products.merge(
    category_names, on="product_category_name", how="left"
)
products_with_category["category"] = products_with_category[
    "product_category_name_english"
].fillna("Other")
payment_per_order = raw_payments.groupby("order_id")["payment_value"].sum().reset_index()

olist = (
    delivered_orders
    .merge(items_per_order, on="order_id", how="left")
    .merge(review_per_order, on="order_id", how="left")
    .merge(
        products_with_category[["product_id", "category"]],
        on="product_id",
        how="left",
    )
    .merge(raw_sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
    .merge(payment_per_order, on="order_id", how="left")
)
olist["total_revenue"] = olist["item_price_total"] + olist["freight_value_total"]
olist = olist.dropna(subset=["delivery_delay_days", "item_price_total"]).copy()

print(f"Analysis dataset: {len(olist):,} delivered orders")
print(
    "Purchase dates: "
    f"{olist['order_purchase_timestamp'].min().date()} to "
    f"{olist['order_purchase_timestamp'].max().date()}"
)
print(f"Categories: {olist['category'].nunique()}")
print(f"Seller states: {olist['seller_state'].nunique()}")

## 3. Management KPI overview

In [ ]:
total_orders = len(olist)
total_revenue = olist["total_revenue"].sum()
on_time_rate = olist["on_time"].mean()
late_orders = int(olist["late"].sum())
avg_late_days = olist.loc[olist["late"].eq(1), "delivery_delay_days"].mean()
avg_review = olist["review_score"].mean()
late_order_revenue = olist.loc[olist["late"].eq(1), "total_revenue"].sum()

kpis = pd.DataFrame(
    {
        "Metric": [
            "Delivered orders",
            "Total order revenue",
            "On-time rate",
            "Late orders",
            "Average delay among late orders",
            "Revenue associated with late orders",
            "Average review score",
        ],
        "Value": [
            f"{total_orders:,}",
            f"R${total_revenue:,.0f}",
            f"{on_time_rate:.1%}",
            f"{late_orders:,} ({1 - on_time_rate:.1%})",
            f"{avg_late_days:.1f} days",
            f"R${late_order_revenue:,.0f}",
            f"{avg_review:.2f} / 5",
        ],
    }
)
kpis

![Management KPI dashboard](../reports/figures/chart1_kpi_dashboard.png)

“Revenue associated with late orders” means the value of affected orders. It is not a measured revenue loss.

## 4. Category performance

In [ ]:
category_performance = (
    summarise_performance(olist, "category", minimum_orders=200)
    .sort_values("on_time_rate", ascending=True)
    .reset_index(drop=True)
)

category_display = category_performance.head(15)
print(
    category_display.to_string(
        index=False,
        formatters={
            "order_count": lambda value: f"{value:,.0f}",
            "on_time_rate": lambda value: f"{value:.1f}%",
            "avg_delay_days": lambda value: f"{value:.2f}",
            "avg_review": lambda value: f"{value:.2f}",
            "total_revenue": lambda value: f"R${value:,.0f}",
        },
    )
)

![Category performance](../reports/figures/chart2_category_performance.png)

I would use these category differences to decide where to investigate first, not as proof that the category caused the delay. Carrier, route, seller, and destination may also matter.

## 5. SQL analysis

DuckDB makes the SQL executable inside the notebook. The business logic is portable, but functions, identifiers, and syntax may need adjustment in PostgreSQL, Snowflake, or Azure SQL.

In [ ]:
db = duckdb.connect()
db.register("olist", olist)

state_scorecard = db.execute(
    '''
    SELECT
        seller_state,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(on_time) * 100, 1) AS on_time_pct,
        ROUND(AVG(CASE WHEN late = 1 THEN delivery_delay_days END), 1) AS avg_late_days,
        ROUND(AVG(review_score), 2) AS avg_review,
        ROUND(SUM(total_revenue), 0) AS revenue
    FROM olist
    WHERE seller_state IS NOT NULL
    GROUP BY seller_state
    HAVING COUNT(order_id) >= 200
    ORDER BY on_time_pct ASC
    '''
).df()

state_scorecard.head(10)

In [ ]:
late_low_review_exposure = db.execute(
    '''
    SELECT
        category,
        COUNT(order_id) AS affected_orders,
        ROUND(SUM(total_revenue), 0) AS revenue_exposure,
        ROUND(AVG(delivery_delay_days), 1) AS avg_late_days,
        ROUND(AVG(review_score), 2) AS avg_review
    FROM olist
    WHERE late = 1
      AND review_score <= 2
      AND category IS NOT NULL
    GROUP BY category
    HAVING COUNT(order_id) >= 10
    ORDER BY revenue_exposure DESC
    LIMIT 10
    '''
).df()

late_low_review_exposure

In [ ]:
quarterly_summary = db.execute(
    '''
    SELECT
        order_quarter,
        COUNT(order_id) AS total_orders,
        ROUND(SUM(total_revenue), 0) AS revenue,
        ROUND(AVG(on_time) * 100, 1) AS on_time_pct,
        ROUND(AVG(delivery_delay_days), 1) AS avg_delay_days,
        ROUND(AVG(review_score), 2) AS avg_review,
        SUM(CASE WHEN late = 1 AND review_score <= 2 THEN 1 ELSE 0 END)
            AS late_low_review_orders
    FROM olist
    GROUP BY order_quarter
    ORDER BY order_quarter
    '''
).df()

quarterly_summary

In [ ]:
seller_scorecard = db.execute(
    '''
    WITH seller_metrics AS (
        SELECT
            seller_id,
            seller_state,
            COUNT(order_id) AS order_count,
            ROUND(AVG(on_time) * 100, 1) AS on_time_pct,
            ROUND(AVG(review_score), 2) AS avg_review,
            ROUND(SUM(total_revenue), 0) AS revenue
        FROM olist
        WHERE seller_id IS NOT NULL
        GROUP BY seller_id, seller_state
        HAVING COUNT(order_id) >= 50
    )
    SELECT *,
        CASE
            WHEN on_time_pct >= 95 AND avg_review >= 4.0 THEN 'Top Performer'
            WHEN on_time_pct < 80 OR avg_review < 3.0 THEN 'Priority Review'
            ELSE 'Core'
        END AS descriptive_tier
    FROM seller_metrics
    ORDER BY on_time_pct ASC
    '''
).df()

seller_scorecard.groupby("descriptive_tier").agg(
    sellers=("seller_id", "count"),
    orders=("order_count", "sum"),
    revenue=("revenue", "sum"),
)

## 6. Regional patterns

![Seller-state performance](../reports/figures/chart3_state_performance.png)

Delivery performance varies across seller states. I would treat this as a starting point for route and carrier analysis rather than conclude that geography is the cause. Customer destination, distance, carrier, shipment type, and seller-level controls are missing here.

## 7. Monthly trends

![Monthly trends](../reports/figures/chart4_monthly_trends.png)

The monthly view can support capacity planning. The first and last partial periods should not be compared directly with complete months.

## 8. Delivery timing and customer reviews

In [ ]:
pearson_correlation = olist["delivery_delay_days"].corr(olist["review_score"])

review_by_timing = pd.DataFrame(
    {
        "Timing": [
            "On time or early",
            "1–3 days late",
            "4–7 days late",
            "8–14 days late",
            "More than 14 days late",
        ],
        "Average review": [
            olist.loc[olist["delivery_delay_days"].le(0), "review_score"].mean(),
            olist.loc[olist["delivery_delay_days"].between(1, 3), "review_score"].mean(),
            olist.loc[olist["delivery_delay_days"].between(4, 7), "review_score"].mean(),
            olist.loc[olist["delivery_delay_days"].between(8, 14), "review_score"].mean(),
            olist.loc[olist["delivery_delay_days"].gt(14), "review_score"].mean(),
        ],
    }
)

print(f"Pearson correlation: {pearson_correlation:.3f}")
print(
    review_by_timing.to_string(
        index=False,
        formatters={"Average review": lambda value: f"{value:.2f}"},
    )
)

![Delivery timing and customer reviews](../reports/figures/chart5_satisfaction_analysis.png)

Late deliveries have lower review scores, with a visible decline after the promised date is missed. Reviews can also reflect product quality, seller service, communication, and other factors, so I do not interpret this as a causal estimate.

## 9. Late-delivery screening benchmark

I define `late = 1` because late orders are the cases the operations team needs to find. The model uses order-time information, class weighting, one-hot encoding, and a threshold selected on validation data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "item_price_total",
    "freight_value_total",
    "item_count",
    "payment_value",
    "promised_lead_days",
    "day_of_week",
    "month_num",
]
categorical_features = ["seller_state", "category"]
feature_columns = numeric_features + categorical_features

model_data = olist[feature_columns + ["late"]].copy()
X = model_data[feature_columns]
y = model_data["late"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val,
)

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)
late_model = Pipeline(
    [
        ("prepare", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2_000,
                random_state=42,
            ),
        ),
    ]
)

late_model.fit(X_train, y_train)
validation_probability = late_model.predict_proba(X_valid)[:, 1]
threshold_result = best_f1_threshold(y_valid, validation_probability)
decision_threshold = threshold_result["threshold"]

test_probability = late_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= decision_threshold).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, test_prediction, average="binary", zero_division=0
)

model_metrics = pd.Series(
    {
        "Late-order prevalence": y_test.mean(),
        "ROC-AUC": roc_auc_score(y_test, test_probability),
        "PR-AUC": average_precision_score(y_test, test_probability),
        "Validation-selected threshold": decision_threshold,
        "Late-order precision": precision,
        "Late-order recall": recall,
        "Late-order F1": f1,
    },
    name="Result",
)
print(model_metrics.to_string(float_format=lambda value: f"{value:.3f}"))

In [ ]:
print("Confusion matrix — rows are actual, columns are predicted")
print(confusion_matrix(y_test, test_prediction))
print()
print(classification_report(y_test, test_prediction, target_names=["On time", "Late"], zero_division=0))

### What the model result means

ROC-AUC measures ranking, not accuracy. Because late orders are uncommon, I also report PR-AUC, precision, and recall.

The first baseline reached ROC-AUC ≈ 0.71 but caught only 3 of 1,307 late orders. The revised evaluation makes that failure visible. A real threshold should reflect the cost of missed delays and false alerts and then be tested on later orders.

## 10. Descriptive satisfaction-threshold analysis

In [ ]:
daily_review = (
    olist.groupby("delivery_delay_days")
    .agg(avg_review=("review_score", "mean"), order_count=("order_id", "count"))
    .reset_index()
)
daily_review = daily_review.loc[
    daily_review["delivery_delay_days"].between(0, 20)
    & daily_review["order_count"].ge(30)
].sort_values("delivery_delay_days")
daily_review["change_from_prior_observed_day"] = daily_review["avg_review"].diff()

largest_drop = daily_review.loc[daily_review["change_from_prior_observed_day"].idxmin()]
print(
    "Largest observed day-to-day mean review decline: "
    f"day {int(largest_drop['delivery_delay_days'])} "
    f"({largest_drop['change_from_prior_observed_day']:.2f} points)"
)

exposure_rows = []
for threshold in (3, 7, 14):
    affected = olist["delivery_delay_days"].gt(threshold)
    exposure_rows.append(
        {
            "Threshold": f"> {threshold} days late",
            "Orders": int(affected.sum()),
            "Revenue exposure": olist.loc[affected, "total_revenue"].sum(),
            "Average review": olist.loc[affected, "review_score"].mean(),
        }
    )

exposure_summary = pd.DataFrame(exposure_rows)
print(
    exposure_summary.to_string(
        index=False,
        formatters={
            "Orders": lambda value: f"{value:,.0f}",
            "Revenue exposure": lambda value: f"R${value:,.0f}",
            "Average review": lambda value: f"{value:.2f}",
        },
    )
)

![Descriptive satisfaction-threshold analysis](../reports/figures/chart7_breakpoint_analysis.png)

The highlighted point is simply the largest change between grouped daily averages. I do not treat it as an exact or causal breakpoint.

## 11. Exploratory seller-state segmentation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

state_performance = (
    olist.groupby("seller_state")
    .agg(
        total_orders=("order_id", "count"),
        on_time_rate=("on_time", "mean"),
        avg_delay=("delivery_delay_days", "mean"),
        avg_review=("review_score", "mean"),
        total_revenue=("total_revenue", "sum"),
    )
    .reset_index()
    .query("total_orders >= 100")
)

cluster_features = state_performance[["on_time_rate", "avg_delay", "avg_review"]]
cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(cluster_features)

silhouette_scores = {}
for number_of_clusters in range(2, min(6, len(state_performance))):
    candidate_model = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=10,
    )
    candidate_labels = candidate_model.fit_predict(cluster_scaled)
    silhouette_scores[number_of_clusters] = silhouette_score(
        cluster_scaled, candidate_labels
    )

optimal_k = max(silhouette_scores, key=silhouette_scores.get)
cluster_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
state_performance["cluster"] = cluster_model.fit_predict(cluster_scaled)

cluster_order = (
    state_performance.groupby("cluster")["on_time_rate"]
    .mean()
    .sort_values(ascending=False)
    .index
)
labels_by_rank = ["Top tier", "Upper-middle", "Lower-middle", "Priority review", "Review"]
cluster_names = {
    cluster: labels_by_rank[rank]
    for rank, cluster in enumerate(cluster_order)
}
state_performance["descriptive_tier"] = state_performance["cluster"].map(cluster_names)

print(f"Silhouette scores: {silhouette_scores}")
print(f"Selected cluster count: {optimal_k}")
state_performance.sort_values("on_time_rate", ascending=False)

![Exploratory seller-state segmentation](../reports/figures/chart8_seller_segmentation.png)

The clusters summarize similarity in three historical metrics. I use them to explore patterns, not as formal supplier ratings.

## 12. Time-series summary

In [ ]:
monthly_trends = (
    olist.groupby("order_month")
    .agg(
        orders=("order_id", "count"),
        revenue=("total_revenue", "sum"),
        on_time_rate=("on_time", "mean"),
        avg_review=("review_score", "mean"),
    )
    .reset_index()
    .sort_values("order_month")
)
monthly_trends["three_month_order_average"] = monthly_trends["orders"].rolling(3).mean()

fuller_months = monthly_trends.loc[monthly_trends["orders"].gt(100)].copy()
print(
    "Peak observed month: "
    f"{fuller_months.loc[fuller_months['orders'].idxmax(), 'order_month']} "
    f"({fuller_months['orders'].max():,.0f} orders)"
)
print(
    "Monthly revenue/on-time correlation: "
    f"{fuller_months['revenue'].corr(fuller_months['on_time_rate']):.3f}"
)
fuller_months.tail()

![Time-series analysis](../reports/figures/chart9_timeseries_analysis.png)

The time series suggests questions for capacity planning, but I would validate them with complete periods and later data before setting targets.

## 13. What I would do next

1. Track both the on-time rate and the length of delays; the 6.8% exception group averaged 10.6 days late.
2. Test proactive communication and escalation after a missed promise date, then measure whether it improves review scores or repeat purchasing.
3. Break category and regional gaps down by route, carrier, destination, and seller before assigning a cause.
4. Use the model only as a screening experiment until recall, precision, and alert costs are acceptable.
5. Separate partial months and validate apparent seasonal patterns with more years of data.

## Limitations and next steps

- Historical data from one marketplace limits generalization.
- Buyer geography, carrier identity, route distance, and shipment attributes would strengthen diagnostics.
- Multi-item and multi-seller orders need more detailed representation.
- Temporal validation is required before any production model claim.
- An intervention test would be needed to estimate causal business impact.

---

[LinkedIn](https://linkedin.com/in/marziyeh-eslamparasti) · [GitHub portfolio](https://github.com/marziyeh-ba)